In [3]:
!pip install --upgrade --force-reinstall srsly

  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/632.3 kB ? eta -:--:--
   ---------------------------------------- 632.3/632.3 kB 3.4 MB/s eta 0:00:00
Using cached catalogue-2.0.10-py3-none-any.whl (17 kB)
  Attempting uninstall: catalogue
    Found existing installation: catalogue 2.0.10
    Uninstalling catalogue-2.0.10:
      Successfully uninstalled catalogue-2.0.10
  Attempting uninstall: srsly
    Found existing installation: srsly 2.5.1
    Uninstalling srsly-2.5.1:
      Successfully uninstalled srsly-2.5.1


In [1]:
!pip uninstall numpy scipy mkl
!pip install numpy scipy

^C
^C


In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [1]:
!pip install gradio PyPDF2 python-docx spacy
!python -m spacy download en_core_web_sm

OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.


In [ ]:
import spacy
from spacy.matcher import Matcher

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

def extract_education(text):
    """Extract education information using spaCy."""
    doc = nlp(text)
    education = []
    for ent in doc.ents:
        if ent.label_ == "ORG" and any(word in ent.text.lower() for word in ["university", "college", "institute"]):
            education.append(ent.text)
    return education

def extract_skills(text):
    """Extract skills using spaCy."""
    doc = nlp(text)
    skills = set()
    for token in doc:
        if token.text.lower() in {"python", "java", "sql", "git", "machine learning", "data visualization"}:
            skills.add(token.text)
    return skills

def extract_experience(text):
    """Extract years of experience using regex."""
    experience_pattern = r"(\d+)\s*(?:years?|yrs?)\s*(?:of)?\s*experience"
    experience_match = re.search(experience_pattern, text, re.IGNORECASE)
    return int(experience_match.group(1)) if experience_match else 0

In [ ]:
def analyze_job_description(job_description):
    """Analyze the job description to determine feature weights."""
    weights = {
        "education": 0.2,
        "experience": 0.3,
        "technical_skills": 0.25,
        "soft_skills": 0.15,
        "work_type": 0.05,
        "location": 0.05
    }

    # Increase weight for experience if "years of experience" is mentioned
    if "years of experience" in job_description.lower():
        weights["experience"] += 0.1
        weights["education"] -= 0.05

    # Increase weight for technical skills if specific tools or languages are mentioned
    if "python" in job_description.lower() or "java" in job_description.lower():
        weights["technical_skills"] += 0.1
        weights["soft_skills"] -= 0.05

    # Normalize weights to ensure they sum to 1
    total_weight = sum(weights.values())
    for key in weights:
        weights[key] /= total_weight

    return weights

In [ ]:
from docx import Document

def extract_text_from_docx(docx_file):
    """Extract text from a DOCX resume."""
    doc = Document(docx_file)
    text = ""
    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"
    return text

In [ ]:
import gradio as gr
import re
import PyPDF2
from docx import Document
import spacy
from spacy.matcher import Matcher

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Mock data for job roles and learning resources
JOB_ROLES = {
    "Software Engineer": {
        "skills": {"Python", "Java", "SQL", "Git"},
        "education": {"degree": "Bachelor's", "field": "Computer Science"},
        "experience": 3,
        "work_type": "Hybrid",
        "location": "New York"
    },
    "Data Scientist": {
        "skills": {"Python", "Machine Learning", "SQL", "Data Visualization"},
        "education": {"degree": "Master's", "field": "Data Science"},
        "experience": 2,
        "work_type": "Remote",
        "location": "Any"
    }
}

LEARNING_RESOURCES = {
    "Python": "https://www.learnpython.org/",
    "Java": "https://www.codecademy.com/learn/learn-java",
    "SQL": "https://www.w3schools.com/sql/",
    "Git": "https://git-scm.com/doc",
    "Machine Learning": "https://www.coursera.org/learn/machine-learning",
    "Data Visualization": "https://www.datacamp.com/courses/data-visualization"
}

def extract_text_from_pdf(resume_file):
    """Extract text from a PDF resume."""
    reader = PyPDF2.PdfReader(resume_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

def extract_text_from_docx(docx_file):
    """Extract text from a DOCX resume."""
    doc = Document(docx_file)
    text = ""
    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"
    return text

def extract_education(text):
    """Extract education information using spaCy."""
    doc = nlp(text)
    education = []
    for ent in doc.ents:
        if ent.label_ == "ORG" and any(word in ent.text.lower() for word in ["university", "college", "institute"]):
            education.append(ent.text)
    return education

def extract_skills(text):
    """Extract skills using spaCy."""
    doc = nlp(text)
    skills = set()
    for token in doc:
        if token.text.lower() in {"python", "java", "sql", "git", "machine learning", "data visualization"}:
            skills.add(token.text)
    return skills

def extract_experience(text):
    """Extract years of experience using regex."""
    experience_pattern = r"(\d+)\s*(?:years?|yrs?)\s*(?:of)?\s*experience"
    experience_match = re.search(experience_pattern, text, re.IGNORECASE)
    return int(experience_match.group(1)) if experience_match else 0

def extract_features(resume_text):
    """Extract features (education, experience, skills, etc.) from resume text."""
    features = {
        "education": extract_education(resume_text),
        "experience": extract_experience(resume_text),
        "skills": extract_skills(resume_text),
        "work_type": "Any",
        "location": "Any"
    }
    return features

def analyze_job_description(job_description):
    """Analyze the job description to determine feature weights."""
    weights = {
        "education": 0.2,
        "experience": 0.3,
        "technical_skills": 0.25,
        "soft_skills": 0.15,
        "work_type": 0.05,
        "location": 0.05
    }

    # Increase weight for experience if "years of experience" is mentioned
    if "years of experience" in job_description.lower():
        weights["experience"] += 0.1
        weights["education"] -= 0.05

    # Increase weight for technical skills if specific tools or languages are mentioned
    if "python" in job_description.lower() or "java" in job_description.lower():
        weights["technical_skills"] += 0.1
        weights["soft_skills"] -= 0.05

    # Normalize weights to ensure they sum to 1
    total_weight = sum(weights.values())
    for key in weights:
        weights[key] /= total_weight

    return weights

def calculate_match_score(resume, job_role, job_description):
    """Calculate a weighted match score based on resume and job description."""
    weights = analyze_job_description(job_description)

    required_skills = JOB_ROLES.get(job_role, {}).get("skills", set())
    required_education = JOB_ROLES.get(job_role, {}).get("education", {})
    required_experience = JOB_ROLES.get(job_role, {}).get("experience", 0)
    required_work_type = JOB_ROLES.get(job_role, {}).get("work_type", "Any")
    required_location = JOB_ROLES.get(job_role, {}).get("location", "Any")

    if not required_skills:
        return 0.0, {}

    # Skill Match
    matched_skills = resume["skills"] & required_skills
    missing_skills = required_skills - matched_skills
    skill_score = (len(matched_skills) / len(required_skills)) * 100 if required_skills else 0

    # Experience Match
    experience_score = min(100, (resume["experience"] / required_experience) * 100) if required_experience else 100

    # Education Match
    education_score = 100 if resume["education"] == required_education else 0

    # Work Type Match
    work_type_score = 100 if resume["work_type"] == required_work_type or required_work_type == "Any" else 0

    # Location Match
    location_score = 100 if resume["location"] == required_location or required_location == "Any" else 0

    # Weighted Score Calculation
    match_percentage = (
        skill_score * weights["technical_skills"] +
        experience_score * weights["experience"] +
        education_score * weights["education"] +
        work_type_score * weights["work_type"] +
        location_score * weights["location"]
    )

    # Generate learning recommendations
    learning_links = [f"{skill}: {LEARNING_RESOURCES.get(skill, 'No course available')}" for skill in missing_skills]

    return round(match_percentage, 2), {
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "learning_links": learning_links,
        "education_match": education_score,
        "experience_match": experience_score,
        "work_type_match": work_type_score,
        "location_match": location_score
    }

def process_resume(resume_file, job_role, job_description):
    """Process the uploaded resume and calculate the match score."""
    # Extract text from the resume
    if resume_file.name.endswith(".pdf"):
        resume_text = extract_text_from_pdf(resume_file)
    elif resume_file.name.endswith(".docx"):
        resume_text = extract_text_from_docx(resume_file)
    else:
        return "Unsupported file format. Please upload a PDF or DOCX file."

    # Extract features from the resume text
    resume_features = extract_features(resume_text)

    # Calculate match score
    match_score, details = calculate_match_score(resume_features, job_role, job_description)

    # Format the output
    output = f"Match Score: {match_score}%\n\n"
    output += "Matched Skills:\n" + "\n".join(details["matched_skills"]) + "\n\n"
    output += "Missing Skills:\n" + "\n".join(details["missing_skills"]) + "\n\n"
    output += "Learning Recommendations:\n" + "\n".join(details["learning_links"]) + "\n\n"
    output += f"Education Match: {details['education_match']}%\n"
    output += f"Experience Match: {details['experience_match']}%\n"
    output += f"Work Type Match: {details['work_type_match']}%\n"
    output += f"Location Match: {details['location_match']}%"

    return output

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("## AI-Driven Pre-Screening Interview Bot")
    with gr.Row():
        resume_input = gr.File(label="Upload Resume (PDF or DOCX)")
        job_role_input = gr.Dropdown(choices=["Software Engineer", "Data Scientist"], label="Select Job Role")
        job_description_input = gr.Textbox(label="Paste Job Description", lines=5)
    submit_button = gr.Button("Calculate Match Score")
    output = gr.Textbox(label="Match Score and Recommendations", lines=15)

    submit_button.click(
        process_resume,
        inputs=[resume_input, job_role_input, job_description_input],
        outputs=output
    )

# Launch the Gradio app
demo.launch(share = True)